<a href="https://colab.research.google.com/github/adebartolo/AnalyticsWithPython/blob/master/sales_discount_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sales Discount Engine

## Purpose
This system assigns personalized discounts (10%, 20%, 30%) to users based on behavior and attributes.

It:
- Uses user signals (LTV, intent, recency, cart value, etc.)
- Applies deterministic randomness (same user → same discount)
- Keeps probabilistic behavior (not rigid rules) for flexibility and experimentation

---

## Value
- Improves conversion efficiency by targeting discounts where they matter most  
- Protects margin by avoiding unnecessary discounts for high-value users  
- Experiment-friendly due to deterministic assignment (reproducible results)  
- Scales easily into batch scoring or production pipelines  
- Balances personalization with controlled randomness  

---

## Pros
- Simple and interpretable logic (especially with normal distribution approach)  
- Easy to tune (via mean shifts or multipliers)  
- Deterministic → consistent user experience across sessions  
- Avoids one-size-fits-all discounting strategies  
- Works without machine learning (strong baseline system)  

---

## Cons
- Heuristic-based (not directly optimized for revenue)  
- Sensitive to tuning (can over-bias if not calibrated)  
- Does not guarantee globally optimal discount per user  
- No feedback loop or learning from outcomes  
- Distribution can drift if input signals are skewed  

---

## Key Takeaway
This is a **smart baseline decision system**:

- Better than random or flat discounts  
- Strong for experimentation and early-stage optimization  
- Still below advanced approaches like:
  - Uplift modeling  
  - Reinforcement learning  
  - Expected revenue optimization  

---

## Sales Discount Logic Inputs

### 1. User Value / LTV Tier
- Historical spend (total + recent)
- Average order value (AOV)
- Purchase frequency  

**Usage:**
- High LTV → 10% discount (protect margin)
- Low LTV / new users → 20–30% (acquisition focus)

---

### 2. Conversion Propensity
- Modeled likelihood to purchase
- Behavioral signals (views, cart adds)

**Usage:**
- High intent → 10%
- Medium intent → 20%
- Low intent → 30%

---

### 3. Recency / Lifecycle Stage
- Days since last purchase
- New vs active vs churned users

**Usage:**
- New users → 20–30%
- Active users → 10–20%
- Lapsed users → 30%

---

### 4. Price Sensitivity
- Past coupon usage
- Discount responsiveness

**Usage:**
- High sensitivity → 20–30%
- Low sensitivity → 10–20%

---

### 5. Cart / Session Context
- Cart value
- Product margin category
- Inventory pressure

**Usage:**
- High cart value → lower discount (protect margin)
- Low cart value → higher discount (boost conversion)

---

### 6. Channel / Acquisition Source
- Paid, organic, email, affiliate

**Usage:**
- Paid → higher discount (optimize CAC)
- Organic → lower discount (protect margin)

---

### 7. Experimentation Controls (tbd)
- A/B test assignment
- Control vs treatment design

**Usage:**
- Maintain control group (no discount) for lift measurement
- Iterate weights based on observed performance

---

### 8. Business Constraints (tbd)
- Margin thresholds
- Promo budget caps
- Inventory availability

In [51]:
import random
import hashlib
from dataclasses import dataclass

In [52]:
@dataclass
class User:
    user_id: str
    ltv: float
    days_since_last_purchase: int
    intent_score: float
    price_sensitivity: float
    cart_value: float
    is_new_user: bool


# ---------------------------
# Deterministic RNG
# ---------------------------
def get_rng(user_id: str):
    seed = int(hashlib.md5(user_id.encode()).hexdigest()[:8], 16)
    return random.Random(seed)


# ---------------------------
# Mean adjustment
# ---------------------------
# Purpose:
# Shift center of distribution slightly based on user traits
def get_mean(user: User):
    mean = 20  # base

    # Low intent → increase discount
    if user.intent_score < 0.3:
        mean += 3
    elif user.intent_score > 0.7:
        mean -= 3

    # High LTV → decrease discount
    if user.ltv > 1000:
        mean -= 2
    elif user.ltv < 200:
        mean += 2

    # Lapsed user → increase discount
    if user.days_since_last_purchase > 90:
        mean += 2

    # Price sensitivity
    if user.price_sensitivity > 0.7:
        mean += 2

    # Large cart → protect margin
    if user.cart_value > 200:
        mean -= 2

    # New user boost
    if user.is_new_user:
        mean += 1

    return mean


# ---------------------------
# Assign discount
# ---------------------------
def assign_discount(user: User):
    rng = get_rng(user.user_id)

    mean = get_mean(user)
    std_dev = 5  # controls "looseness"

    # Draw from normal distribution
    score = rng.gauss(mean, std_dev)

    # Bucket into discounts
    if score < 15:
        return 10
    elif score < 25:
        return 20
    else:
        return 30

# Test results
if __name__ == "__main__":
    users = [
        User("user_1", 50, 10, 0.2, 0.9, 30, True),
        User("user_2", 1200, 5, 0.8, 0.2, 300, False),
        User("user_3", 300, 120, 0.1, 0.8, 40, False),
        User("user_4", 800, 60, 0.5, 0.5, 150, False),
        User("user_5", 100, 200, 0.2, 0.9, 20, True),
        User("user_6", 2000, 2, 0.9, 0.1, 500, False),
        User("user_7", 150, 30, 0.3, 0.7, 80, False),
        User("user_8", 400, 10, 0.6, 0.4, 120, False),
        User("user_9", 50, 300, 0.1, 0.9, 25, True),
        User("user_10", 900, 15, 0.7, 0.3, 220, False),
    ]

    print("User Results:\n")

    results = {10: 0, 20: 0, 30: 0}

    for user in users:
        discount = assign_discount(user)
        results[discount] += 1

        print(f"{user.user_id}: {discount}%")

    print("\nDistribution Summary:")
    for k, v in results.items():
        print(f"{k}%: {v}")

User Results:

user_1: 30%
user_2: 10%
user_3: 20%
user_4: 30%
user_5: 30%
user_6: 10%
user_7: 20%
user_8: 30%
user_9: 30%
user_10: 20%

Distribution Summary:
10%: 2
20%: 3
30%: 5


In [53]:
# Test random users
for i in range(10):
    user.user_id = f"user_{i}"
    print(user.user_id, assign_discount(user))

user_0 10
user_1 20
user_2 20
user_3 10
user_4 20
user_5 20
user_6 10
user_7 10
user_8 20
user_9 20


In [54]:
if __name__ == "__main__":
    import random

    # Generate many users for better distribution view
    users = [
        User(
            user_id=f"user_{i}",
            ltv=random.randint(0, 2000),
            days_since_last_purchase=random.randint(0, 365),
            intent_score=random.random(),
            price_sensitivity=random.random(),
            cart_value=random.randint(10, 500),
            is_new_user=random.choice([True, False])
        )
        for i in range(1000)
    ]

    results = {10: 0, 20: 0, 30: 0}

    for user in users:
        discount = assign_discount(user)
        results[discount] += 1

    # ---------------------------
    # Histogram Output
    # ---------------------------
    print("\nHistogram (Distribution of Discounts):\n")

    total = sum(results.values())

    for discount in sorted(results.keys()):
        count = results[discount]
        pct = count / total

        bar = "█" * int(pct * 50)  # scale to max width of 50 chars

        print(f"{discount}% | {bar} ({count}, {pct:.1%})")


Histogram (Distribution of Discounts):

10% | ████████ (164, 16.4%)
20% | █████████████████████████████ (593, 59.3%)
30% | ████████████ (243, 24.3%)
